In [1]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

asplanned_file = next(
    name for name in uploaded_names
    if "asplanned_ifc_elements_extracted" in name.lower()
)

planned_file = next(
    name for name in uploaded_names
    if "planned_events_cleaned" in name.lower()
)

asbuilt_file = next(
    name for name in uploaded_names
    if "asbuilt_events_cleaned" in name.lower()
)

asplanned_bim = pd.read_csv(asplanned_file)
planned = pd.read_csv(planned_file)
asbuilt = pd.read_csv(asbuilt_file)

print("As-planned BIM:", asplanned_bim.shape)
print("Planned event log:", planned.shape)
print("As-built event log:", asbuilt.shape)

Saving asbuilt_events_cleaned.csv to asbuilt_events_cleaned.csv
Saving asplanned_ifc_elements_extracted.csv to asplanned_ifc_elements_extracted.csv
Saving planned_events_cleaned.csv to planned_events_cleaned.csv
As-planned BIM: (3505, 9)
Planned event log: (7166, 12)
As-built event log: (3661, 13)


In [2]:
# Standardize GUID formatting
for dataframe in [asplanned_bim, planned, asbuilt]:
    dataframe["GUID"] = (
        dataframe["GUID"]
        .astype("string")
        .str.strip()
    )

# Create unique GUID sets
bim_guids = set(
    asplanned_bim["GUID"].dropna()
)

planned_guids = set(
    planned["GUID"].dropna()
)

asbuilt_guids = set(
    asbuilt["GUID"].dropna()
)

# Find matched and unmatched GUIDs
planned_matched = planned_guids.intersection(
    bim_guids
)

planned_unmatched = planned_guids.difference(
    bim_guids
)

asbuilt_matched = asbuilt_guids.intersection(
    bim_guids
)

asbuilt_unmatched = asbuilt_guids.difference(
    bim_guids
)

guid_coverage = pd.DataFrame({
    "Dataset": ["Planned", "As-built"],
    "Unique_Log_GUIDs": [
        len(planned_guids),
        len(asbuilt_guids)
    ],
    "Matched_GUIDs": [
        len(planned_matched),
        len(asbuilt_matched)
    ],
    "Unmatched_GUIDs": [
        len(planned_unmatched),
        len(asbuilt_unmatched)
    ]
})

guid_coverage["GUID_Match_Percent"] = (
    guid_coverage["Matched_GUIDs"]
    / guid_coverage["Unique_Log_GUIDs"]
    * 100
).round(2)

display(guid_coverage)

,Dataset,Unique_Log_GUIDs,Matched_GUIDs,Unmatched_GUIDs,GUID_Match_Percent
0,Planned,3505,3505,0,100.0
1,As-built,2358,2358,0,100.0


In [3]:
# Prefix BIM columns so they remain distinguishable
bim_for_join = asplanned_bim.rename(
    columns={
        column: f"BIM_{column}"
        for column in asplanned_bim.columns
        if column != "GUID"
    }
).copy()

# Create a standardized BIM class field
bim_for_join["BIM_IfcClass_Clean"] = (
    bim_for_join["BIM_IfcClass"]
    .astype("string")
    .str.strip()
    .str.replace(r"^Ifc", "", regex=True)
    .str.replace(" ", "", regex=False)
    .str.lower()
)

# Merge planned events with BIM information
planned_integrated = planned.merge(
    bim_for_join,
    on="GUID",
    how="left",
    validate="many_to_one",
    indicator="MatchStatus"
)

# Merge as-built events with BIM information
asbuilt_integrated = asbuilt.merge(
    bim_for_join,
    on="GUID",
    how="left",
    validate="many_to_one",
    indicator="MatchStatus"
)

print("Planned integrated shape:", planned_integrated.shape)
print(
    planned_integrated["MatchStatus"]
    .value_counts()
)

print("\nAs-built integrated shape:", asbuilt_integrated.shape)
print(
    asbuilt_integrated["MatchStatus"]
    .value_counts()
)

Planned integrated shape: (7166, 22)
MatchStatus
both          7166
left_only        0
right_only       0
Name: count, dtype: int64

As-built integrated shape: (3661, 23)
MatchStatus
both          3661
left_only        0
right_only       0
Name: count, dtype: int64


In [4]:
planned_integrated["Class_Exact_Match"] = (
    planned_integrated["IfcClass_Clean"]
    == planned_integrated["BIM_IfcClass_Clean"]
).fillna(False)

asbuilt_integrated["Class_Exact_Match"] = (
    asbuilt_integrated["IfcClass_Clean"]
    == asbuilt_integrated["BIM_IfcClass_Clean"]
).fillna(False)

print("Planned IFC-class consistency:")
print(
    planned_integrated["Class_Exact_Match"]
    .value_counts()
)

print("\nAs-built IFC-class consistency:")
print(
    asbuilt_integrated["Class_Exact_Match"]
    .value_counts()
)

Planned IFC-class consistency:
Class_Exact_Match
True     5940
False    1226
Name: count, dtype: Int64

As-built IFC-class consistency:
Class_Exact_Match
True    3661
Name: count, dtype: Int64


In [5]:
planned_mismatches = planned_integrated[
    planned_integrated["Class_Exact_Match"] == False
].copy()

# Make missing values visible
planned_mismatches["Log_Class"] = (
    planned_mismatches["IfcClass_Clean"]
    .fillna("missing")
)

planned_mismatches["BIM_Class"] = (
    planned_mismatches["BIM_IfcClass_Clean"]
    .fillna("missing")
)

mismatch_summary = (
    planned_mismatches
    .groupby(
        ["Log_Class", "BIM_Class"],
        dropna=False
    )
    .agg(
        Record_Count=("GUID", "size"),
        Unique_GUIDs=("GUID", "nunique"),
        Unique_Tasks=("TaskID", "nunique")
    )
    .reset_index()
    .sort_values(
        "Record_Count",
        ascending=False
    )
)

display(mismatch_summary)

,Log_Class,BIM_Class,Record_Count,Unique_GUIDs,Unique_Tasks
1,missing,buildingelementpart,574,277,6
4,wall,wallstandardcase,436,232,20
3,roof,slab,130,62,6
2,pipesegment,buildingelementproxy,60,60,1
0,distributionelement,buildingelementproxy,26,13,2


In [6]:
mismatch_guid_set = set(
    planned_mismatches["GUID"].dropna()
)

asbuilt_guid_set = set(
    asbuilt_integrated["GUID"].dropna()
)

print(
    "Unique GUIDs with planned class mismatch:",
    len(mismatch_guid_set)
)

print(
    "Mismatch GUIDs appearing in as-built data:",
    len(
        mismatch_guid_set.intersection(
            asbuilt_guid_set
        )
    )
)

print(
    "Mismatch GUIDs absent from as-built data:",
    len(
        mismatch_guid_set.difference(
            asbuilt_guid_set
        )
    )
)

Unique GUIDs with planned class mismatch: 644
Mismatch GUIDs appearing in as-built data: 455
Mismatch GUIDs absent from as-built data: 189


In [7]:
display(mismatch_summary.head(20))

,Log_Class,BIM_Class,Record_Count,Unique_GUIDs,Unique_Tasks
1,missing,buildingelementpart,574,277,6
4,wall,wallstandardcase,436,232,20
3,roof,slab,130,62,6
2,pipesegment,buildingelementproxy,60,60,1
0,distributionelement,buildingelementproxy,26,13,2


In [8]:
planned_class_map = (
    planned_integrated[
        [
            "GUID",
            "IfcClass_Clean",
            "BIM_IfcClass_Clean"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "IfcClass_Clean":
                "Planned_Log_Class"
        }
    )
)

asbuilt_class_map = (
    asbuilt_integrated[
        [
            "GUID",
            "IfcClass_Clean"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "IfcClass_Clean":
                "Asbuilt_Log_Class"
        }
    )
)

class_comparison = planned_class_map.merge(
    asbuilt_class_map,
    on="GUID",
    how="left"
)

class_comparison["Planned_vs_Asbuilt_Match"] = (
    class_comparison["Planned_Log_Class"]
    == class_comparison["Asbuilt_Log_Class"]
)

display(
    class_comparison[
        class_comparison[
            "Planned_Log_Class"
        ] != class_comparison[
            "BIM_IfcClass_Clean"
        ]
    ].head(20)
)

,GUID,Planned_Log_Class,BIM_IfcClass_Clean,Asbuilt_Log_Class,Planned_vs_Asbuilt_Match
6,1qSygSJAf1FwTO1RCAcXhe,wall,wallstandardcase,wallstandardcase,False
17,0kS6wEmzbD2va78wYRyOFc,wall,wallstandardcase,wallstandardcase,False
96,0R69HoRWz0C9s$TSI4WT3Z,wall,wallstandardcase,wallstandardcase,False
101,2gSzfhTzjCkutSogmAtpkP,wall,wallstandardcase,wallstandardcase,False
102,2eUGgnxUXF0RJQZi7ey1aP,wall,wallstandardcase,wallstandardcase,False
110,3kp8yBgj5BreEI464Vtha8,wall,wallstandardcase,wallstandardcase,False
111,1EWidDAVz66hylOSXB1Kjk,wall,wallstandardcase,wallstandardcase,False
113,2D_6Rg4dz6$PEfPb1fvJzF,wall,wallstandardcase,wallstandardcase,False
115,397vy$qeXCzhtQ8tWx7ugf,wall,wallstandardcase,wallstandardcase,False
119,2mUBHykl91Ue0vPS8ztnuL,wall,wallstandardcase,wallstandardcase,False


In [10]:
# Short references to the two class columns
log_class = planned_integrated["IfcClass_Clean"]
bim_class = planned_integrated["BIM_IfcClass_Clean"]

# Default category
planned_integrated["Class_Relationship"] = (
    "Other mismatch"
)

# Exact matches
planned_integrated.loc[
    log_class == bim_class,
    "Class_Relationship"
] = "Exact match"

# Missing planned-log classification
planned_integrated.loc[
    log_class.isna(),
    "Class_Relationship"
] = "Missing class in planned log"

# Wall versus WallStandardCase
planned_integrated.loc[
    (log_class == "wall")
    & (bim_class == "wallstandardcase"),
    "Class_Relationship"
] = "Compatible wall subclass"

# Roof versus slab
planned_integrated.loc[
    (log_class == "roof")
    & (bim_class == "slab"),
    "Class_Relationship"
] = "Functional roof-slab difference"

# MEP elements represented as proxies
planned_integrated.loc[
    log_class.isin([
        "pipesegment",
        "distributionelement"
    ])
    & (bim_class == "buildingelementproxy"),
    "Class_Relationship"
] = "Generic proxy representation"

# Use the IFC model as the authoritative class source
planned_integrated["Canonical_IfcClass"] = bim_class

# As-built records already matched exactly
asbuilt_integrated["Class_Relationship"] = (
    "Exact match"
)

asbuilt_integrated["Canonical_IfcClass"] = (
    asbuilt_integrated["BIM_IfcClass_Clean"]
)

classification_summary = (
    planned_integrated["Class_Relationship"]
    .value_counts()
    .rename_axis("Class_Relationship")
    .reset_index(name="Records")
)

display(classification_summary)

,Class_Relationship,Records
0,Exact match,5940
1,Missing class in planned log,574
2,Compatible wall subclass,436
3,Functional roof-slab difference,130
4,Generic proxy representation,86


In [12]:
# MatchStatus is no longer needed in the final analytical files
planned_final = planned_integrated.drop(
    columns=["MatchStatus"],
    errors="ignore"
)

asbuilt_final = asbuilt_integrated.drop(
    columns=["MatchStatus"],
    errors="ignore"
)

planned_final.to_csv(
    "planned_bim_integrated.csv",
    index=False
)

asbuilt_final.to_csv(
    "asbuilt_bim_integrated.csv",
    index=False
)

classification_summary.to_csv(
    "ifc_class_interoperability_summary.csv",
    index=False
)

In [13]:
from google.colab import files

files.download("planned_bim_integrated.csv")
files.download("asbuilt_bim_integrated.csv")
files.download("ifc_class_interoperability_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>